# From First Principles to Modern Large Language Models

**Author:** Paul Leyva

An unbroken chain of derivation from set theory to modern LLMs.


## Table of contents

### A — Foundations
- Chapter 1: Sets, functions, logic, proofs
- Chapter 2: Numbers, sequences, limits, completeness
- Chapter 3: Continuity, univariate differentiation, chain rule
- Chapter 4: Multivariate calculus: partials, gradients, Jacobians
- Chapter 5: Linear algebra I: vector spaces, basis, linear maps
- Chapter 6: Linear algebra II: inner products, norms, eigenvalues, SVD
- Chapter 7: Convexity and optimization; gradient descent convergence

### B — Probability and Information
- Chapter 8: Probability foundations: sample spaces, sigma-algebras, Kolmogorov axioms
- Chapter 9: Random variables, distributions, CDF/PMF/PDF
- Chapter 10: Expectation, variance, covariance; Jensen's inequality
- Chapter 11: Information theory: self-information, entropy, cross-entropy, KL
- Chapter 12: Statistical inference: likelihood, MLE, ERM, bias-variance

### C — Stochastic Optimization
- Chapter 13: SGD: stochastic-approximation theorem; mini-batching; convergence sketch
- Chapter 14: Momentum, RMSProp, AdamW: derivation and bias-correction proof

### D — Neural Networks
- Chapter 15: MLPs as compositional functions; universal approximation
- Chapter 16: Activation functions: ReLU/GELU/softmax with derivatives
- Chapter 17: Loss functions: MSE, cross-entropy; gradients from first principles
- Chapter 18: Backpropagation: chain rule applied; reverse-mode AD as a graph algorithm

### E — Sequence Models and Attention
- Chapter 19: Embeddings: token to vector; lookup as a linear map; weight tying
- Chapter 20: RNN intuition; vanishing-gradient proof; why we need attention
- Chapter 21: Scaled dot-product attention: derivation, softmax-temperature analysis
- Chapter 22: Multi-head attention: parallel heads as concat-then-project; complexity
- Chapter 23: Transformer block: residual + LayerNorm/RMSNorm + FFN + attention; gradient-flow argument
- Chapter 24: Positional encoding: sinusoidal derivation, RoPE construction

### F — Pre-training
- Chapter 25: Causal masking; next-token prediction loss as MLE on the empirical distribution
- Chapter 26: Tokenization: BPE algorithm; greedy merge correctness
- Chapter 27: Pre-training pipeline: AdamW + warmup + cosine decay + gradient clipping; tiny-GPT training run

### G — Post-training
- Chapter 28: SFT, RLHF (PPO/GRPO), and DPO; train + post-train a tiny GPT



# Block A — Foundations


# Chapter 1 — Sets, functions, logic, proofs

We need a precise grammar for membership, functions, and proof before we can define real numbers (Ch. 5), linear maps (Ch. 8), probability (Ch. 15), or token embeddings (Ch. 19).

**Key definitions.** A *set* $S$ is a collection of distinct elements; $x \in S$ means $x$ is an element of $S$. The *power set* is $\mathcal{P}(S) = \{T : T \subset S\}$. A *function* $f : A \to B$ assigns each $a \in A$ exactly one $f(a) \in B$.


In [ ]:
from itertools import chain, combinations

def power_set(S):
    S = list(S)
    return [set(c) for r in range(len(S) + 1) for c in combinations(S, r)]

S = {'a', 'b', 'c'}
P = power_set(S)
for T in P:
    print(sorted(T))
print('|P(S)| =', len(P), '   2**|S| =', 2 ** len(S))
assert len(P) == 2 ** len(S), 'power-set cardinality identity failed'


## De Morgan's laws

For $A, B \subset U$:
$$(A \cup B)^c = A^c \cap B^c, \qquad (A \cap B)^c = A^c \cup B^c.$$


In [ ]:
U = set(range(1, 9))
A = {1, 3, 5, 7}
B = {2, 3, 5, 7}

comp = lambda X: U - X

lhs1, rhs1 = comp(A | B), comp(A) & comp(B)
lhs2, rhs2 = comp(A & B), comp(A) | comp(B)

print('(A u B)^c =', sorted(lhs1), '  A^c n B^c =', sorted(rhs1))
print('(A n B)^c =', sorted(lhs2), '  A^c u B^c =', sorted(rhs2))
assert lhs1 == rhs1 and lhs2 == rhs2, 'De Morgan failed'
print('Both De Morgan identities verified.')


## Injection, surjection, bijection

$f : A \to B$ is *injective* iff $f(a_1) = f(a_2) \Rightarrow a_1 = a_2$, *surjective* iff every $b \in B$ has a preimage, *bijective* iff both.


In [ ]:
def is_injective(f, A):
    seen = {}
    for a in A:
        b = f[a]
        if b in seen:
            return False
        seen[b] = a
    return True

def is_surjective(f, A, B):
    return set(f[a] for a in A) == set(B)

def is_bijective(f, A, B):
    return is_injective(f, A) and is_surjective(f, A, B)

A = [0, 1, 2, 3]
B = [0, 1, 2, 3, 4]
f = {0: 1, 1: 3, 2: 0, 3: 4}   # injective into B, not surjective
print('f =', f)
print('injective?', is_injective(f, A))
print('surjective onto B?', is_surjective(f, A, B))
print('bijective A -> B?', is_bijective(f, A, B))

# A bijection A -> A
g = {0: 2, 1: 0, 2: 3, 3: 1}
print('\ng =', g)
print('bijective A -> A?', is_bijective(g, A, A))


## Induction

Claim: $\sum_{k=1}^{n} k = n(n+1)/2$.

*Base:* $n = 1$: LHS $= 1 =$ RHS. *Step:* assume $S(n) = n(n+1)/2$; then
$S(n+1) = S(n) + (n+1) = n(n+1)/2 + (n+1) = (n+1)(n+2)/2$.


In [ ]:
import numpy as np

def S_direct(n):
    return sum(range(1, n + 1))

def S_closed(n):
    return n * (n + 1) // 2

# Direct check up to n = 20
for n in range(1, 21):
    assert S_direct(n) == S_closed(n), f'mismatch at n={n}'
print('Direct verification 1..20: OK')

# Numerical induction-step check: S_closed(n+1) - S_closed(n) == n+1
ns = np.arange(1, 1001)
lhs = np.array([S_closed(n + 1) - S_closed(n) for n in ns])
rhs = ns + 1
assert np.array_equal(lhs, rhs), 'induction step failed'
print('Base case S(1) =', S_closed(1))
print('Induction step S(n+1) - S(n) = n+1 verified for n = 1..1000')


## Connection to LLMs

A vocabulary $\mathcal{V}$ is a finite set of tokens. The embedding map $E : \mathcal{V} \to \mathbb{R}^d$ is a function; its lookup-table implementation requires the index $\mathcal{V} \to \{0, \ldots, |\mathcal{V}|-1\}$ to be a **bijection**. We revisit this in Chapter 19.


In [ ]:
import numpy as np
np.random.seed(0)

vocab = ['<bos>', '<eos>', 'the', 'cat', 'sat', 'on', 'mat']
V = len(vocab)
d = 4

tok2id = {tok: i for i, tok in enumerate(vocab)}
id2tok = {i: tok for tok, i in tok2id.items()}

# Bijectivity of vocabulary index
assert len(set(tok2id.values())) == V                  # injective
assert set(tok2id.values()) == set(range(V))           # surjective onto {0,...,V-1}
print('vocabulary index is a bijection V <-> {0,...,V-1}')

E = np.random.randn(V, d).astype(np.float32)
print('embedding matrix shape:', E.shape)
for tok in ['cat', 'mat']:
    print(f'E[{tok!r}] =', E[tok2id[tok]])


# Chapter 2 — Numbers, sequences, limits, completeness

We build $\mathbb{N} \subset \mathbb{Z} \subset \mathbb{Q} \subset \mathbb{R}$ and isolate the **completeness axiom** of $\mathbb{R}$: every nonempty subset that is bounded above has a supremum in $\mathbb{R}$. This single axiom is what powers every convergence theorem we will need later for SGD and Adam.


## Convergence: $\varepsilon$–$N$ on $S_n = \sum_{k=1}^n 1/k^2 \to \pi^2/6$

We compute partial sums, the gap $|S_n - \pi^2/6|$, and for each $\varepsilon$ the smallest $N$ such that $n \geq N \Rightarrow |S_n - \pi^2/6| < \varepsilon$.


In [ ]:
import numpy as np

target = np.pi**2 / 6
n_max = 2_000_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
partial = np.cumsum(1.0 / ks**2)

for n in [1, 5, 10, 100, 1_000, 10_000, 100_000]:
    print(f'S_{n:>7d} = {partial[n-1]:.10f}   gap = {abs(partial[n-1] - target):.2e}')

gap = np.abs(partial - target)
for eps in [1e-2, 1e-4, 1e-6]:
    idx = np.argmax(gap < eps)
    # Verify monotone: from idx onward, gap stays < eps (true here since gap ~ 1/n).
    assert gap[idx] < eps
    print(f'eps = {eps:.0e}   smallest N = {idx + 1}')


## Cauchy criterion

A real sequence is Cauchy iff for every $\varepsilon > 0$ there exists $N$ with $|a_m - a_n| < \varepsilon$ for all $m, n \geq N$. We test $a_n = 1/n$ (Cauchy) against $b_n = (-1)^n$ (not Cauchy).


In [ ]:
import numpy as np

def is_cauchy(seq, eps):
    """Return smallest N such that sup_{m,n>=N} |seq[m]-seq[n]| < eps, or None."""
    seq = np.asarray(seq, dtype=np.float64)
    L = len(seq)
    for N in range(L):
        tail = seq[N:]
        if tail.max() - tail.min() < eps:
            return N
    return None

a = 1.0 / np.arange(1, 5001)
b = (-1.0) ** np.arange(5000)

for eps in [1e-1, 1e-2, 1e-3]:
    Na = is_cauchy(a, eps)
    Nb = is_cauchy(b, eps)
    print(f'eps = {eps:.0e}   a_n=1/n: N = {Na}    b_n=(-1)^n: N = {Nb}')


## $\sqrt{2} \notin \mathbb{Q}$ and Newton's method

**Proof recap.** If $\sqrt{2} = p/r$ in lowest terms, then $p^2 = 2r^2$ forces $p$ even, then $r$ even, contradicting $\gcd(p,r)=1$.

So $\sqrt{2}$ lives in $\mathbb{R} \setminus \mathbb{Q}$ — and the Newton iteration $x_{n+1} = (x_n + 2/x_n)/2$ produces a Cauchy sequence of rationals whose limit is $\sqrt{2}$, witnessing why we needed completeness in the first place.


In [ ]:
import numpy as np

x = 1.0
true = np.sqrt(2.0)
print(f'{ "n":>3}  {"x_n":>20}  {"|x_n - sqrt(2)|":>18}')
for n in range(11):
    print(f'{n:>3}  {x:>20.16f}  {abs(x - true):>18.2e}')
    x = 0.5 * (x + 2.0 / x)


## Bounded monotone convergence: telescoping series

$a_n = \sum_{k=1}^n \tfrac{1}{k(k+1)} = 1 - \tfrac{1}{n+1}$ is monotone increasing and bounded above by $1$. By Theorem 2.8 it must converge — and indeed to $\sup_n a_n = 1$.


In [ ]:
import numpy as np

n_max = 100_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
terms = 1.0 / (ks * (ks + 1.0))
a = np.cumsum(terms)

monotone = bool(np.all(np.diff(a) >= 0))
bounded = bool(np.all(a <= 1.0))
print(f'monotone increasing: {monotone}')
print(f'bounded above by 1 : {bounded}')
for n in [1, 10, 100, 1_000, 10_000, 100_000]:
    print(f'a_{n:>6d} = {a[n-1]:.10f}   gap to 1 = {1 - a[n-1]:.2e}')
print(f'sup_n a_n (numerical) = {a.max():.12f}')


## Forward link

When we prove SGD converges in Chapter 13, the core lemma is: the loss sequence $(L(\theta_t))$ is monotone decreasing in expectation and bounded below by $0$, hence convergent — by **exactly** Theorem 2.8 of this chapter.


## Continuity, differentiation, chain rule

We numerically explore the four anchors of Chapter 3:
1. $\varepsilon$-$\delta$ continuity at a point.
2. Derivative as a limit (forward difference).
3. Chain rule.
4. Mean value theorem.

**Recall:** $f$ is continuous at $a$ iff $\forall\,\varepsilon>0\;\exists\,\delta>0:\;|x-a|<\delta \Rightarrow |f(x)-f(a)|<\varepsilon$.


In [ ]:
import numpy as np
np.random.seed(0)

# Numerical eps-delta certificate for f(x) = x^2 at a = 2.
# Strategy: for each eps, binary-search the largest delta in (0, 1] for which
# sup_{|x-a|<delta} |f(x)-f(a)| < eps holds (sampled densely).

def f(x):
    return x * x

a = 2.0
fa = f(a)

def violation(delta, n=4001):
    xs = np.linspace(a - delta, a + delta, n)
    return np.max(np.abs(f(xs) - fa))

def find_delta(eps, lo=0.0, hi=1.0, iters=60):
    # halving search: largest hi with violation(hi) < eps
    if violation(hi) < eps:
        return hi
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if violation(mid) < eps:
            lo = mid
        else:
            hi = mid
    return lo

print(f'{"eps":>10} {"delta found":>15} {"sup|f(x)-f(a)|":>20}  ok?')
for eps in (1e-1, 1e-2, 1e-3):
    d = find_delta(eps)
    sup = violation(d)
    print(f'{eps:>10.0e} {d:>15.8e} {sup:>20.8e}  {sup < eps}')


## Derivative as a limit

$f'(a) = \lim_{h\to 0} \frac{f(a+h)-f(a)}{h}$.

The forward-difference truncation error for smooth $f$ is $O(h)$ (Taylor expansion). Below we plot it on a log-log axis if matplotlib is available; otherwise we print a table.


In [ ]:
import numpy as np

# Forward-difference derivative of sin at a grid of points.
xs = np.linspace(0.1, np.pi - 0.1, 200)
true = np.cos(xs)

hs = np.array([10.0 ** k for k in range(-1, -13, -1)])
errors = np.array([np.max(np.abs((np.sin(xs + h) - np.sin(xs)) / h - true)) for h in hs])

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.loglog(hs, errors, 'o-', label='forward diff error')
    ax.loglog(hs, hs, 'k--', alpha=0.5, label='O(h) reference')
    ax.set_xlabel('h'); ax.set_ylabel('max error'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout(); fig.savefig('fd_error.png', dpi=120)
    print('Saved fd_error.png')
except Exception as e:
    print(f'(matplotlib unavailable: {e}) -- printing table instead')

print(f'{"h":>12} {"max |fd - cos|":>20}')
for h, err in zip(hs, errors):
    print(f'{h:>12.1e} {err:>20.6e}')


## Chain rule

If $g$ is differentiable at $a$ and $f$ is differentiable at $g(a)$, then
$$ (f\circ g)'(a) = f'(g(a)) \cdot g'(a). $$

**Sketch (Carath\'eodory):** define $\phi(y)=(f(y)-f(b))/(y-b)$ for $y\ne b$ and $\phi(b)=f'(b)$. Then $\phi$ is continuous at $b=g(a)$ and $f(y)-f(b)=\phi(y)(y-b)$ identically. Substitute $y=g(x)$, divide by $x-a$, take $x\to a$.

We verify on $f(x)=\sin(x^2)$ at $x=1$: analytic value is $\cos(1)\cdot 2 \approx 1.0806$.


In [ ]:
import numpy as np

x = 1.0
F = lambda t: np.sin(t * t)

analytic = np.cos(x * x) * 2.0 * x  # f'(g(x)) * g'(x) with f=sin, g=x^2

# Central difference is O(h^2) and minimizes truncation+roundoff at h ~ 1e-5.
h = 1e-5
numeric = (F(x + h) - F(x - h)) / (2 * h)

print(f'analytic   f\'(g(x)) * g\'(x) = {analytic:.12f}')
print(f'numeric    central diff h=1e-5 = {numeric:.12f}')
print(f'abs error                       = {abs(analytic - numeric):.3e}')
assert abs(analytic - numeric) < 1e-7, 'chain rule check failed'
print('chain rule verified to <1e-7')


## Mean value theorem

If $f$ is continuous on $[a,b]$ and differentiable on $(a,b)$, then $\exists c \in (a,b)$ with
$$ f'(c) = \frac{f(b)-f(a)}{b-a}. $$

We find such a $c$ for $f(x)=x^3-2x$ on $[0,2]$ by bisection on $f'(x)-\text{slope}$.


In [ ]:
import numpy as np

def f(x):  return x**3 - 2*x
def fp(x): return 3*x**2 - 2

a, b = 0.0, 2.0
slope = (f(b) - f(a)) / (b - a)
g = lambda x: fp(x) - slope

# fp is continuous; g(0) = -2 - 1 = -3, g(2) = 10 - 1 = 9, sign change => bisection works.
lo, hi = a, b
assert g(lo) * g(hi) < 0
for _ in range(80):
    mid = 0.5 * (lo + hi)
    if g(lo) * g(mid) <= 0:
        hi = mid
    else:
        lo = mid
c = 0.5 * (lo + hi)

print(f'slope (f(b)-f(a))/(b-a) = {slope:.10f}')
print(f'witness c              = {c:.10f}')
print(f'f\'(c)                  = {fp(c):.10f}')
print(f'|f\'(c) - slope|        = {abs(fp(c) - slope):.3e}')
# Closed form: 3c^2 - 2 = 2  =>  c = sqrt(4/3)
print(f'closed form sqrt(4/3)   = {np.sqrt(4/3):.10f}')


## Multivariate calculus: partials, gradients, Jacobians

We move from $f:\mathbb{R}\to\mathbb{R}$ (Chapter 3) to $f:\mathbb{R}^n\to\mathbb{R}^m$. The right notion of differentiability is **Fréchet**: there exists a linear $L$ with $\|f(\mathbf{a}+\mathbf{h})-f(\mathbf{a})-L\mathbf{h}\| = o(\|\mathbf{h}\|)$. The matrix of $L$ is the **Jacobian** $J_f(\mathbf{a}) \in \mathbb{R}^{m\times n}$, whose $j$-th column is $\partial f/\partial x_j(\mathbf{a})$ (Theorem 4.5). When $m=1$, $\nabla f = J_f^\top$.

Below we verify the key facts numerically with finite differences.

In [ ]:
import numpy as np
np.random.seed(0)

# f: R^2 -> R, f(x,y) = x^2 y + sin(x+y)
def f(x, y):
    return x**2 * y + np.sin(x + y)

# Analytic gradient: df/dx = 2xy + cos(x+y), df/dy = x^2 + cos(x+y)
def grad_f_analytic(x, y):
    return np.array([2*x*y + np.cos(x+y), x**2 + np.cos(x+y)])

def grad_f_numeric(x, y, h=1e-6):
    dx = (f(x+h, y) - f(x-h, y)) / (2*h)
    dy = (f(x, y+h) - f(x, y-h)) / (2*h)
    return np.array([dx, dy])

x0, y0 = 1.0, 2.0
ga = grad_f_analytic(x0, y0)
gn = grad_f_numeric(x0, y0)
print(f'analytic grad at (1,2): {ga}')
print(f'numeric  grad at (1,2): {gn}')
print(f'max abs error: {np.max(np.abs(ga-gn)):.2e}')

## Jacobians by column

For $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^m$, the $j$-th column of $J_f(\mathbf{a})$ is $\partial \mathbf{f}/\partial x_j(\mathbf{a})$, computable as the central difference $(\mathbf{f}(\mathbf{a}+h\mathbf{e}_j)-\mathbf{f}(\mathbf{a}-h\mathbf{e}_j))/(2h)$. We test on $\mathbf{f}(x,y,z)=(xyz,\ x^2+\sin y+e^z)$, whose analytic Jacobian is

$$J_f = \begin{pmatrix} yz & xz & xy \\ 2x & \cos y & e^z \end{pmatrix}.$$

In [ ]:
import numpy as np

def F(v):
    x, y, z = v
    return np.array([x*y*z, x**2 + np.sin(y) + np.exp(z)])

def J_analytic(v):
    x, y, z = v
    return np.array([
        [y*z,      x*z,      x*y],
        [2*x,      np.cos(y), np.exp(z)],
    ])

def J_numeric(F, v, h=1e-6):
    v = np.asarray(v, dtype=float)
    n = v.size
    cols = []
    for j in range(n):
        ej = np.zeros(n); ej[j] = 1.0
        cols.append((F(v + h*ej) - F(v - h*ej)) / (2*h))
    return np.stack(cols, axis=1)

v0 = np.array([1.0, 0.5, -0.3])
Ja = J_analytic(v0)
Jn = J_numeric(F, v0)
print('analytic J:'); print(Ja)
print('numeric  J:'); print(Jn)
print(f'max abs error: {np.max(np.abs(Ja-Jn)):.2e}')

## Multivariate chain rule

Theorem 4.7: $J_{f\circ g}(\mathbf{a}) = J_f(g(\mathbf{a}))\, J_g(\mathbf{a})$. Take $g(t)=(\cos t,\sin t)$ and $f(x,y)=x^2+y^2$. Then $f\circ g \equiv 1$, so $(f\circ g)'(t)=0$ for every $t$. The chain rule must produce the same answer.

In [ ]:
import numpy as np

def g(t):
    return np.array([np.cos(t), np.sin(t)])

def gprime(t):
    return np.array([-np.sin(t), np.cos(t)])  # column vector (R -> R^2 has 2x1 Jacobian)

def grad_f(p):
    x, y = p
    return np.array([2*x, 2*y])  # row of J_f

ts = np.linspace(0, 2*np.pi, 7)
for t in ts:
    direct = 0.0  # f(g(t)) = 1, derivative is 0
    chain  = float(grad_f(g(t)) @ gprime(t))   # 1x2 @ 2x1
    print(f't={t:6.3f}  direct={direct:+.2e}  chain-rule={chain:+.2e}')

## Schwarz / Clairaut: equality of mixed partials

For $C^2$ functions, $\partial_x\partial_y f = \partial_y\partial_x f$ (Theorem 4.8). Numerically, both can be approximated by the second-order central difference

$$\partial_x\partial_y f(a,b) \approx \frac{f(a+h,b+k)-f(a+h,b-k)-f(a-h,b+k)+f(a-h,b-k)}{4hk}.$$

We test on $f(x,y) = x^3 y^2 + \sin(xy)$ at $(1,1)$. Analytically,
$\partial_y f = 2x^3 y + x\cos(xy)$, so $\partial_x\partial_y f = 6x^2 y + \cos(xy) - xy\sin(xy)$, which at $(1,1)$ is $6 + \cos(1) - \sin(1)$.

In [ ]:
import numpy as np

def fxy(x, y):
    return x**3 * y**2 + np.sin(x*y)

def mixed_partial(F, a, b, h=1e-3, k=1e-3):
    return (F(a+h, b+k) - F(a+h, b-k) - F(a-h, b+k) + F(a-h, b-k)) / (4*h*k)

a, b = 1.0, 1.0
dxdy = mixed_partial(fxy, a, b)
dydx = mixed_partial(lambda y, x: fxy(x, y), b, a)  # swap roles
analytic = 6*a**2*b + np.cos(a*b) - a*b*np.sin(a*b)

print(f'numeric  d/dx d/dy f at (1,1): {dxdy:.10f}')
print(f'numeric  d/dy d/dx f at (1,1): {dydx:.10f}')
print(f'analytic value             : {analytic:.10f}')
print(f'|dxdy - dydx| = {abs(dxdy-dydx):.2e}')
print(f'|num - analytic| = {abs(dxdy-analytic):.2e}')

## Connection to LLMs

A transformer is a composition $F = F_L\circ\cdots\circ F_1$. Theorem 4.7 says the gradient of the loss with respect to layer-$\ell$ parameters is a product of Jacobians from the loss back to that layer. Backpropagation never materializes those matrices: it propagates a *row vector* $\mathbf{v}^\top$ right-to-left via vector–Jacobian products (VJPs), one per layer, each at $O(\text{forward cost})$. We will derive reverse-mode autodiff formally in Chapter 18.

# Chapter 5 — Linear algebra I: vector spaces, basis, linear maps

Every transformer layer is a linear map between finite-dimensional real vector spaces, framed by bias terms and nonlinearities. Before attention (Ch. 21) or embeddings (Ch. 19), we need vector spaces, bases, dimension, kernels, images, and the rank–nullity theorem.

**Eight axioms of a vector space $V$ over a field $\mathbb{F}$.** For $u, v, w \in V$, $a, b \in \mathbb{F}$:

1. $(u+v)+w = u+(v+w)$
2. $u+v = v+u$
3. $\exists\, 0 \in V$ with $v+0=v$
4. $\forall v\,\exists (-v)$ with $v+(-v)=0$
5. $a(u+v) = au + av$
6. $(a+b)v = av + bv$
7. $(ab)v = a(bv)$
8. $1\cdot v = v$

The canonical example is $V = \mathbb{R}^d$ over $\mathbb{F} = \mathbb{R}$, which is exactly the embedding space used by language models with hidden dimension $d$.

In [ ]:
import numpy as np

def is_linearly_independent(vectors):
    """Vectors is a list/array of row vectors. Independent iff rank == count."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M)) == M.shape[0]

def dim_span(vectors):
    """Dimension of the span of a list of vectors."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M))

S = [(1, 0, 0), (0, 1, 0), (1, 1, 0)]
print('vectors:', S)
print('linearly independent?', is_linearly_independent(S))
print('dim of span:', dim_span(S))
assert dim_span(S) == 2, 'expected 2 since (1,1,0) = (1,0,0) + (0,1,0)'

## Basis and dimension

A **basis** of $V$ is a linearly independent spanning set. By the **Steinitz exchange lemma**, every basis of a finite-dimensional $V$ has the same cardinality, called $\dim V$. Below we compute the rank of a $4 \times 6$ matrix (the dimension of its column span / image) and extract a basis of its **null space** (kernel) via the right singular vectors of $A$ associated to zero singular values.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
print('A =\n', A)

U, sigma, Vt = np.linalg.svd(A)
print('singular values:', np.round(sigma, 4))

# Numerical rank: count singular values above tolerance.
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
print('rank(A) =', rank)

# Right singular vectors (rows of Vt) corresponding to zero singular values
# span the null space. There are n - rank of them, where n = 6.
n = A.shape[1]
null_basis = Vt[rank:]  # shape (n - rank, n)
print('dim ker(A) =', null_basis.shape[0])

# Verify A @ v == 0 for each null-space basis vector v.
for i, v in enumerate(null_basis):
    Av = A @ v
    print(f'A v_{i} norm = {np.linalg.norm(Av):.2e}')
    assert np.allclose(Av, 0, atol=1e-10)

## Linear maps and matrix representation

A linear map $T: V \to W$ satisfies $T(au+bv) = aTu + bTv$. In bases $(e_j)$ of $V = \mathbb{R}^n$ and $(f_i)$ of $W = \mathbb{R}^m$, the matrix $A \in \mathbb{R}^{m\times n}$ has $j$-th column equal to the coordinates of $T(e_j)$. If we change basis on $V = W = \mathbb{R}^n$ via an invertible $P$, the same map $T$ is represented in the new basis by $\tilde{A} = P^{-1} A P$. Numerically: for any $v \in \mathbb{R}^n$ we should have $A v = P\,\tilde{A}\,P^{-1} v$.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randn(3, 3)
# Build a random invertible P. Re-draw if singular (essentially never happens).
while True:
    P = np.random.randn(3, 3)
    if abs(np.linalg.det(P)) > 1e-6:
        break
P_inv = np.linalg.inv(P)
A_tilde = P_inv @ A @ P

v = np.random.randn(3)
lhs = A @ v
rhs = P @ A_tilde @ P_inv @ v
print('A v       =', lhs)
print('P A~ Pi v =', rhs)
print('max abs diff:', np.max(np.abs(lhs - rhs)))
assert np.allclose(lhs, rhs)

## Rank–nullity

**Theorem.** For $T: V \to W$ with $\dim V < \infty$, $\dim \ker T + \dim \mathrm{im}\,T = \dim V$. Equivalently, for $A \in \mathbb{R}^{m\times n}$, $\mathrm{rank}(A) + \dim \ker(A) = n$. We verify this for the $4 \times 6$ matrix above (so the sum should equal $6$).

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
U, sigma, Vt = np.linalg.svd(A)
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
nullity = A.shape[1] - rank  # by definition of SVD null-space basis
print(f'rank(A)    = {rank}')
print(f'nullity(A) = {nullity}')
print(f'sum        = {rank + nullity}')
print(f'n (cols)   = {A.shape[1]}')
assert rank + nullity == A.shape[1], 'rank-nullity violated'

## Connection to LLMs

A transformer with hidden dimension $d$ operates in $\mathbb{R}^d$. The query/key/value projections in attention (Ch. 21) are linear maps $\mathbb{R}^d \to \mathbb{R}^{d_k}$. Token embeddings (Ch. 19) are a linear map $\mathbb{R}^{|\mathcal{V}|} \to \mathbb{R}^d$ applied to a one-hot input. LoRA constrains weight *updates* to a low-rank subspace, an explicit application of $\dim \mathrm{im}\,T \le \min(m, n)$. Rank–nullity will reappear whenever we count free parameters or degrees of freedom.

## Inner products, norms, and Cauchy-Schwarz

The dot product on $\mathbb{R}^n$ is $\langle x,y\rangle=\sum_i x_iy_i$, and its induced norm is $\|x\|=\sqrt{\langle x,x\rangle}$. Cauchy-Schwarz says $|\langle x,y\rangle|\le\|x\|\,\|y\|$. We numerically verify this on 50 random pairs in $\mathbb{R}^{10}$ by computing the ratio $|\langle x,y\rangle|/(\|x\|\|y\|)$ -- it must always be $\le 1$.


In [ ]:
import numpy as np

np.random.seed(0)
n_pairs, dim = 50, 10
ratios = []
for _ in range(n_pairs):
    x = np.random.randn(dim)
    y = np.random.randn(dim)
    inner = float(np.dot(x, y))
    nx = float(np.linalg.norm(x))
    ny = float(np.linalg.norm(y))
    ratios.append(abs(inner) / (nx * ny))

ratios = np.array(ratios)
print(f'pairs tested        : {n_pairs}')
print(f'max  |<x,y>|/(|x||y|): {ratios.max():.6f}')
print(f'mean |<x,y>|/(|x||y|): {ratios.mean():.6f}')
assert ratios.max() <= 1.0 + 1e-12, 'Cauchy-Schwarz violated!'
print('Cauchy-Schwarz holds for all 50 pairs.')


## Eigenvalues of symmetric matrices

The spectral theorem says every real symmetric matrix $A$ has an orthonormal eigenbasis: $A=Q\Lambda Q^\top$. Below we build a symmetric $5\times5$ matrix $A=M+M^\top$, diagonalize with `np.linalg.eigh` (which exploits symmetry), and verify both $A v_i=\lambda_i v_i$ for every $i$ and $V^\top V=I$ (orthonormal eigenvectors).


In [ ]:
import numpy as np

np.random.seed(0)
M = np.random.randn(5, 5)
A = M + M.T
assert np.allclose(A, A.T)

eigvals, V = np.linalg.eigh(A)
print('eigenvalues:', np.round(eigvals, 6))

max_eig_residual = 0.0
for i in range(A.shape[0]):
    lhs = A @ V[:, i]
    rhs = eigvals[i] * V[:, i]
    max_eig_residual = max(max_eig_residual, float(np.linalg.norm(lhs - rhs)))
print(f'max ||A v_i - lambda_i v_i|| : {max_eig_residual:.2e}')

ortho_err = float(np.linalg.norm(V.T @ V - np.eye(5)))
print(f'||V^T V - I||_F              : {ortho_err:.2e}')

assert max_eig_residual < 1e-10
assert ortho_err < 1e-10
print('Spectral theorem verified numerically.')


## Singular value decomposition

Every $A\in\mathbb{R}^{m\times n}$ admits an SVD $A=U\Sigma V^\top$ with $U,V$ orthogonal and $\Sigma$ diagonal with nonnegative entries. We build a random $5\times3$ matrix, run `np.linalg.svd`, and reconstruct $A$ to high precision.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

Sigma = np.zeros_like(A)
Sigma[:len(s), :len(s)] = np.diag(s)
A_reco = U @ Sigma @ Vt

print('singular values        :', np.round(s, 6))
print('U shape, Sigma shape, Vt shape:', U.shape, Sigma.shape, Vt.shape)
print(f'||A - U Sigma V^T||_F  : {np.linalg.norm(A - A_reco):.2e}')
print(f'||U^T U - I||_F        : {np.linalg.norm(U.T @ U - np.eye(5)):.2e}')
print(f'||V V^T - I||_F        : {np.linalg.norm(Vt @ Vt.T - np.eye(3)):.2e}')

assert np.linalg.norm(A - A_reco) < 1e-10
print('SVD reconstruction verified.')


## Eckart-Young: best low-rank approximation

Eckart-Young says the best rank-$k$ approximation of $A$ in Frobenius norm is the truncated SVD $A_k=\sum_{i=1}^k\sigma_i u_i v_i^\top$, with squared error $\sum_{i>k}\sigma_i^2$. We compute rank-1 and rank-2 truncations of the same $5\times3$ matrix and observe a strict monotone decrease in error, matching the predicted tail sum of squared singular values.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

errors = {}
for k in (1, 2, 3):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    err = float(np.linalg.norm(A - A_k))
    predicted = float(np.sqrt(np.sum(s[k:] ** 2)))
    errors[k] = err
    print(f'rank {k}: ||A - A_k||_F = {err:.6f}   predicted sqrt(sum sigma_i^2 for i>k) = {predicted:.6f}')

assert errors[1] >= errors[2] >= errors[3]
assert errors[3] < 1e-10
print('Frobenius error decreases monotonically with rank, as Eckart-Young predicts.')


# Chapter 7 — Convexity and gradient descent

Why convex? Because it is the **only** setting where we can write down honest convergence rates for $x_{t+1} = x_t - \eta \nabla f(x_t)$. Real transformer losses are non-convex (Chapter 13, 14, 27), but the convex rates supply the vocabulary we use to reason about them: $L$-smoothness, condition number $\kappa = L/\mu$, contraction.

**Definitions.** $f$ is *convex* iff $f(t x + (1-t) y) \le t f(x) + (1-t) f(y)$. It is *$L$-smooth* iff $\|\nabla f(x) - \nabla f(y)\| \le L\|x - y\|$. It is *$\mu$-strongly convex* iff $f(y) \ge f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{\mu}{2}\|y - x\|^2$.


In [ ]:
import numpy as np

# A strongly convex quadratic f(x, y) = (x - 1)^2 + 2 (y + 1)^2
# Hessian = diag(2, 4), so mu = 2, L = 4, kappa = 2, minimum at x* = (1, -1).
def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2

def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

x_star = np.array([1.0, -1.0])

# Numerical convexity check: midpoint inequality on random pairs.
np.random.seed(0)
violations = 0
for _ in range(2000):
    a, b = np.random.randn(2) * 3, np.random.randn(2) * 3
    t = np.random.rand()
    if f(t * a + (1 - t) * b) > t * f(a) + (1 - t) * f(b) + 1e-9:
        violations += 1
print(f'Convexity violations in 2000 samples: {violations}')

# Contour plot, with table fallback if matplotlib is unavailable.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    xs = np.linspace(-2, 4, 80); ys = np.linspace(-4, 2, 80)
    X, Y = np.meshgrid(xs, ys)
    Z = (X - 1.0)**2 + 2.0 * (Y + 1.0)**2
    plt.figure(figsize=(5, 4))
    plt.contour(X, Y, Z, levels=20)
    plt.scatter([1], [-1], c='red', label='x*')
    plt.title('f(x,y) = (x-1)^2 + 2(y+1)^2'); plt.legend()
    plt.savefig('ch07_contour.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved contour to ch07_contour.png')
except Exception as e:
    print(f'matplotlib unavailable ({e}); printing slice table:')
    for x in [-1, 0, 1, 2, 3]:
        row = [f(np.array([x, y])) for y in [-3, -2, -1, 0, 1]]
        print(f'x={x:+d}: ' + '  '.join(f'{v:6.2f}' for v in row))


## Descent lemma and the $1/L$ step

**Descent lemma.** $L$-smoothness implies $f(y) \le f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{L}{2}\|y - x\|^2$. Plugging $y = x - \tfrac{1}{L}\nabla f(x)$ yields
$$f(x_{t+1}) \le f(x_t) - \tfrac{1}{2L}\|\nabla f(x_t)\|^2,$$
i.e. **monotone descent**. For $\mu$-strongly convex $f$ this strengthens to $\|x_{t+1} - x^*\|^2 \le (1 - \mu/L)\|x_t - x^*\|^2$, a *geometric* contraction.


In [ ]:
import numpy as np

def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])
x_star = np.array([1.0, -1.0])

# GD on the strongly convex quadratic with eta = 1/L.
L = 4.0; mu = 2.0; eta = 1.0 / L
x = np.array([3.5, 1.5])
history = []
for t in range(200):
    history.append(np.linalg.norm(x - x_star)**2)
    x = x - eta * grad_f(x)
history.append(np.linalg.norm(x - x_star)**2)

predicted_factor = 1.0 - mu / L  # 0.5
print(f'Predicted contraction per step: {predicted_factor}')
for t in [0, 1, 2, 5, 10, 20, 50, 100, 200]:
    pred = history[0] * predicted_factor**t
    print(f't={t:3d}  ||x_t - x*||^2 = {history[t]:.3e}   predicted bound = {pred:.3e}')

ratios = [history[t+1] / history[t] for t in range(50) if history[t] > 1e-30]
print(f'Median empirical per-step ratio over first 50 steps: {np.median(ratios):.4f}')


## $O(1/T)$ rate without strong convexity

If $f$ is $L$-smooth and convex but **not** $\mu$-strongly convex, the rate degrades from geometric to $f(x_T) - f^* \le \tfrac{L \|x_0 - x^*\|^2}{2T}$. We exhibit this on a least-squares problem $f(x) = \|Ax - b\|^2$ where $A \in \mathbb{R}^{20 \times 10}$ has a tiny smallest singular value: in the slow direction the effective $\mu$ is essentially zero.


In [ ]:
import numpy as np

np.random.seed(0)
U, _ = np.linalg.qr(np.random.randn(20, 20))
V, _ = np.linalg.qr(np.random.randn(10, 10))
sigma = np.linspace(1.0, 1e-3, 10)  # smallest singular value 1e-3 -> tiny mu
S = np.zeros((20, 10)); np.fill_diagonal(S, sigma)
A = U @ S @ V.T
b = np.random.randn(20)

x_star_ls, *_ = np.linalg.lstsq(A, b, rcond=None)
f_star = float(np.linalg.norm(A @ x_star_ls - b)**2)

L_ls = 2.0 * (sigma.max()**2)
print(f'sigma_max={sigma.max():.4f}, sigma_min={sigma.min():.4f}, L={L_ls:.4f}')

x = np.zeros(10); eta = 1.0 / L_ls
gaps = []
for t in range(1000):
    r = A @ x - b
    g = 2.0 * (A.T @ r)
    x = x - eta * g
    gaps.append(float(np.linalg.norm(A @ x - b)**2) - f_star)

Ts = [1, 2, 5, 10, 50, 100, 500, 1000]
print('   T       f(x_T)-f*        bound L||x0-x*||^2/(2T)')
C = L_ls * float(np.linalg.norm(x_star_ls)**2) / 2.0
for T in Ts:
    print(f'  {T:4d}    {gaps[T-1]:.3e}      {C/T:.3e}')

tail_T = np.arange(100, 1001)
tail_g = np.array(gaps[99:1000])
tail_g = np.maximum(tail_g, 1e-20)
slope, intercept = np.polyfit(np.log(tail_T), np.log(tail_g), 1)
print(f'log-log slope on T in [100, 1000]: {slope:.3f}  (theory predicts ~ -1)')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.loglog(np.arange(1, 1001), np.maximum(gaps, 1e-20), label='f(x_T) - f*')
    plt.loglog(np.arange(1, 1001), C / np.arange(1, 1001), '--', label='L||x0-x*||^2/(2T)')
    plt.xlabel('T'); plt.ylabel('suboptimality'); plt.legend(); plt.title('GD on ill-conditioned LS')
    plt.savefig('ch07_rate.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved log-log rate plot to ch07_rate.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Strong convexity: $\eta = 1/L$ vs. the optimal $\eta = 2/(L + \mu)$

For a quadratic with eigenvalues in $[\mu, L]$ the per-step contraction with $\eta = 1/L$ is $1 - \mu/L$, while with the optimal $\eta = 2/(L + \mu)$ it improves to $((\kappa - 1)/(\kappa + 1))^2$ where $\kappa = L/\mu$. Both rates are linear; the optimal one has a strictly smaller constant.


In [ ]:
import numpy as np
L = 4.0; mu = 2.0
x_star = np.array([1.0, -1.0])
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

def run(eta, T=40):
    x = np.array([3.5, 1.5])
    err = [np.linalg.norm(x - x_star)**2]
    for _ in range(T):
        x = x - eta * grad_f(x)
        err.append(np.linalg.norm(x - x_star)**2)
    return err

eta_basic = 1.0 / L
eta_opt = 2.0 / (L + mu)
h_basic = run(eta_basic)
h_opt = run(eta_opt)
kappa = L / mu
rate_basic = 1.0 - mu / L
rate_opt = ((kappa - 1.0) / (kappa + 1.0))**2
print(f'theory: eta=1/L contracts by {rate_basic:.3f}/step; eta=2/(L+mu) by {rate_opt:.3f}/step')
print(f'  t   ||x_t-x*||^2 (1/L)    ||x_t-x*||^2 (2/(L+mu))')
for t in [0, 5, 10, 20, 40]:
    print(f'  {t:3d}     {h_basic[t]:.3e}             {h_opt[t]:.3e}')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.semilogy(h_basic, label='eta = 1/L')
    plt.semilogy(h_opt, label='eta = 2/(L+mu) (optimal)')
    plt.xlabel('t'); plt.ylabel('||x_t - x*||^2'); plt.legend(); plt.title('Step-size comparison')
    plt.savefig('ch07_steps.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved step-size comparison to ch07_steps.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Forward to the LLM chapters

Transformer training is non-convex, but the same skeleton recurs: a descent-lemma-style inequality (now in expectation) plus a contraction or telescoping argument. SGD (Chapter 13) replaces $(\star)$ with $\mathbb{E}[f(x_{t+1})] \le f(x_t) - \tfrac{\eta}{2}\|\nabla f(x_t)\|^2 + \tfrac{L \eta^2 \sigma^2}{2}$. AdamW (Chapter 14) builds preconditioners on top. The pre-training pipeline (Chapter 27) uses warmup precisely because the *local* $L$ is large early in training, exactly the regime where the descent lemma forbids a large step.


# Block B — Probability and Information


## Probability foundations

A **probability space** is a triple $(\Omega, \mathcal{F}, \mathbb{P})$ where

- $\Omega$ is a non-empty *sample space*;
- $\mathcal{F} \subseteq 2^{\Omega}$ is a *$\sigma$-algebra* (contains $\Omega$, closed under complements and countable unions);
- $\mathbb{P}: \mathcal{F} \to [0, 1]$ satisfies the **Kolmogorov axioms**: non-negativity, $\mathbb{P}(\Omega) = 1$, and countable additivity for disjoint events.

We instantiate the smallest non-trivial example: two fair dice with $\Omega = \{1,\dots,6\}^2$, $\mathcal{F} = 2^{\Omega}$, and uniform $\mathbb{P}(A) = |A|/36$.


In [ ]:
import numpy as np
from itertools import product

# Build Omega = {1,...,6}^2 explicitly as a frozenset of tuples.
Omega = frozenset(product(range(1, 7), repeat=2))
assert len(Omega) == 36

def P(A):
    """Uniform probability measure on Omega: P(A) = |A| / |Omega|."""
    A = set(A)
    assert A.issubset(Omega), 'A must be a subset of Omega'
    return len(A) / len(Omega)

# (K2) normalization
print('P(Omega) =', P(Omega))

# (K3) finite additivity on two disjoint events:
# A = first die equals 1; B = first die equals 2 -- disjoint.
A = {w for w in Omega if w[0] == 1}
B = {w for w in Omega if w[0] == 2}
assert A.isdisjoint(B)
print('P(A) + P(B) =', P(A) + P(B), '   P(A union B) =', P(A | B))
assert abs((P(A) + P(B)) - P(A | B)) < 1e-12


### Inclusion-exclusion and the union bound

For any events $A, B$,
$$\mathbb{P}(A \cup B) = \mathbb{P}(A) + \mathbb{P}(B) - \mathbb{P}(A \cap B).$$

For any countable family,
$$\mathbb{P}\!\left(\bigcup_n A_n\right) \leq \sum_n \mathbb{P}(A_n).$$

Equality fails as soon as the events overlap.


In [ ]:
# Inclusion-exclusion: A = first die = 6, B = second die = 6.
A = {w for w in Omega if w[0] == 6}
B = {w for w in Omega if w[1] == 6}
lhs = P(A | B)
rhs = P(A) + P(B) - P(A & B)
print(f'P(A cup B) = {lhs:.6f}   P(A) + P(B) - P(A cap B) = {rhs:.6f}')
assert abs(lhs - rhs) < 1e-12

# Union bound on three events:
# A1 = first die >= 5; A2 = second die >= 5; A3 = sum >= 10.
A1 = {w for w in Omega if w[0] >= 5}
A2 = {w for w in Omega if w[1] >= 5}
A3 = {w for w in Omega if w[0] + w[1] >= 10}
union = A1 | A2 | A3
print(f'P(union) = {P(union):.6f}   sum P(Ai) = {P(A1) + P(A2) + P(A3):.6f}')
assert P(union) <= P(A1) + P(A2) + P(A3) + 1e-12


### Conditional probability and Bayes' theorem

For $\mathbb{P}(B) > 0$, $\mathbb{P}(A \mid B) = \mathbb{P}(A \cap B) / \mathbb{P}(B)$ and
$$\mathbb{P}(A \mid B) = \frac{\mathbb{P}(B \mid A)\, \mathbb{P}(A)}{\mathbb{P}(B)}.$$

**Disease testing.** Prior $\mathbb{P}(D) = 0.001$, sensitivity $\mathbb{P}(+\mid D) = 0.99$, specificity $\mathbb{P}(-\mid D^c) = 0.95$. We compute $\mathbb{P}(D \mid +)$ analytically and verify by Monte Carlo.


In [ ]:
import numpy as np

prior = 0.001
sens  = 0.99    # P(+ | D)
spec  = 0.95    # P(- | not D), so false-positive rate = 1 - spec = 0.05
fpr   = 1 - spec

# Law of total probability: P(+) = P(+|D)P(D) + P(+|notD)P(notD)
p_pos = sens * prior + fpr * (1 - prior)
# Bayes: P(D | +)
p_d_given_pos = sens * prior / p_pos
print(f'Analytic P(D | +) = {p_d_given_pos:.6f}')

# Monte Carlo verification.
np.random.seed(0)
n = 10_000
diseased = np.random.rand(n) < prior
test_pos = np.where(
    diseased,
    np.random.rand(n) < sens,    # given disease, true-positive
    np.random.rand(n) < fpr,     # given no disease, false-positive
)
n_pos = test_pos.sum()
if n_pos > 0:
    mc = (diseased & test_pos).sum() / n_pos
else:
    mc = float('nan')
print(f'Monte Carlo P(D | +)  = {mc:.6f}   (n_pos = {int(n_pos)} of {n})')
print('Note: with prior 0.001 and n=10000, the MC estimate is noisy but order-of-magnitude consistent.')


### Independence

Events $A, B$ are **independent** iff $\mathbb{P}(A \cap B) = \mathbb{P}(A)\mathbb{P}(B)$. This is a *property of the measure*, not a property of the sets.

On the two-dice space, the events "first die = 6" and "second die = 6" are independent (the dice are physically uncoupled in the uniform measure). The events "first die = 6" and "sum = 12" are *not* independent: knowing the sum is 12 forces both dice to be 6.


In [ ]:
A = {w for w in Omega if w[0] == 6}            # first die = 6
B = {w for w in Omega if w[1] == 6}            # second die = 6
C = {w for w in Omega if w[0] + w[1] == 12}    # sum = 12

print('Independence test for (A, B):')
print(f'  P(A) * P(B)    = {P(A) * P(B):.6f}')
print(f'  P(A cap B)     = {P(A & B):.6f}   --> independent\n')

print('Independence test for (A, C):')
print(f'  P(A) * P(C)    = {P(A) * P(C):.6f}')
print(f'  P(A cap C)     = {P(A & C):.6f}   --> NOT independent')

assert abs(P(A & B) - P(A) * P(B)) < 1e-12
assert abs(P(A & C) - P(A) * P(C)) > 1e-6


### Looking ahead

A causal language model (Chapter 25) defines, for each context $c$, a probability measure on the finite vocabulary $\mathcal{V}$ via the softmax output. Every claim in this chapter -- additivity, the union bound, Bayes, total probability, continuity -- transfers verbatim to that measure. Sampling, beam search, nucleus truncation, and importance-weighted training are all operations on a Kolmogorov probability space.


## Random Variables, Distributions, CDF / PMF / PDF

A random variable $X:\Omega\to\mathbb{R}$ pushes the probability $\mathbb{P}$ on $(\Omega,\mathcal{F})$ to a distribution $\mu_X$ on $\mathbb{R}$. We will (1) build a discrete RV (categorical) by softmaxing fixed logits, (2) compute its PMF and CDF, and (3) sample from it via inverse-CDF.


In [ ]:
import numpy as np

np.random.seed(0)

# Categorical with K=5 outcomes from fixed logits (think: tiny LM head).
logits = np.array([2.0, 1.0, 0.5, -0.5, 0.0])
exp_z = np.exp(logits - logits.max())  # numerical stability
pmf = exp_z / exp_z.sum()

print('PMF :', np.round(pmf, 4))
print('Sum :', pmf.sum())  # must be 1 (Theorem 9.2)

cdf = np.cumsum(pmf)
print('CDF :', np.round(cdf, 4))

# Inverse-CDF sampling (Theorem 9.4): draw U ~ Uniform[0,1], output min k with CDF[k] >= U.
n = 10_000
u = np.random.rand(n)
samples = np.searchsorted(cdf, u, side='left')

empirical = np.bincount(samples, minlength=5) / n
print('Empirical :', np.round(empirical, 4))
print('True PMF  :', np.round(pmf, 4))
print('Max |emp - true| :', np.max(np.abs(empirical - pmf)))


### Continuous distributions and densities

For a continuous RV, $\mathbb{P}(X\in A) = \int_A f_X(x)\,dx$. The standard normal has $f_X(x)=\tfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$. Theorem 9.2 forces $\int_{\mathbb{R}} f_X = 1$. We verify by Riemann sum and build the CDF by cumulative sum.


In [ ]:
import numpy as np

def standard_normal_pdf(x):
    return np.exp(-0.5 * x * x) / np.sqrt(2 * np.pi)

# Truncate to [-8, 8]: the tail mass beyond is < 1e-15.
x_grid = np.linspace(-8.0, 8.0, 16_001)
dx = x_grid[1] - x_grid[0]
f_vals = standard_normal_pdf(x_grid)

total_mass = np.sum(f_vals) * dx  # midpoint/Riemann sum
print(f'Numerical integral of f_X : {total_mass:.10f}  (target 1)')

# CDF via cumulative sum of f_vals * dx.
F_vals = np.cumsum(f_vals) * dx
for x in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    idx = int(np.searchsorted(x_grid, x))
    print(f'F_X({x:+.1f}) = {F_vals[idx]:.6f}')


### Change of variables

Theorem 9.3: if $Y = g(X)$ with $g$ strictly monotone $C^1$, then $f_Y(y) = f_X(g^{-1}(y))\,|(g^{-1})'(y)|$.

Take $X\sim\mathrm{Uniform}[0,1]$ so $f_X = 1$ on $[0,1]$, and $g(x) = -\ln x$ on $(0,1]$. Then $g^{-1}(y) = e^{-y}$, $(g^{-1})'(y) = -e^{-y}$, and
$$f_Y(y) = 1 \cdot |-e^{-y}| = e^{-y},\quad y \geq 0,$$
i.e. $Y\sim\mathrm{Exp}(1)$. We confirm with a histogram of 10000 samples.


In [ ]:
import numpy as np

np.random.seed(0)

x_samples = np.random.rand(10_000)         # X ~ U[0,1]
y_samples = -np.log(x_samples)             # Y = g(X) = -ln X

# Empirical histogram on [0, 6]
edges = np.linspace(0, 6, 25)
counts, _ = np.histogram(y_samples, bins=edges)
centers = 0.5 * (edges[:-1] + edges[1:])
widths = np.diff(edges)
emp_density = counts / (counts.sum() * widths)
true_density = np.exp(-centers)            # f_Y(y) = e^{-y}

print(' y center   empirical f_Y   true f_Y   |diff|')
for c, e, t in zip(centers[:8], emp_density[:8], true_density[:8]):
    print(f'  {c:5.3f}      {e:8.4f}     {t:8.4f}   {abs(e-t):.4f}')

print(f'\nSample mean of Y : {y_samples.mean():.4f}  (true E[Y] = 1)')


### Inverse-CDF sampling, revisited

We re-run inverse-CDF sampling on the categorical from cell 2 and watch the empirical PMF converge to the true PMF as $n$ grows. This is exactly how a language model converts a softmax row into a token id (modulo temperature/top-$k$/top-$p$): cumulative-sum the probabilities, draw $U\sim U[0,1]$, output the first index whose cumulative probability $\geq U$.


In [ ]:
import numpy as np

np.random.seed(0)

logits = np.array([2.0, 1.0, 0.5, -0.5, 0.0])
exp_z = np.exp(logits - logits.max())
pmf = exp_z / exp_z.sum()
cdf = np.cumsum(pmf)

def inverse_cdf_sample(cdf, n, rng):
    u = rng.random(n)
    return np.searchsorted(cdf, u, side='left')

rng = np.random.default_rng(0)
for n in [100, 1_000, 10_000, 100_000]:
    s = inverse_cdf_sample(cdf, n, rng)
    emp = np.bincount(s, minlength=len(pmf)) / n
    err = np.max(np.abs(emp - pmf))
    print(f'n = {n:>7}  max |emp - true| = {err:.4f}')

print('\nTrue PMF:', np.round(pmf, 4))


# Chapter 10 — Expectation, Variance, Covariance, Jensen

We compress a distribution into summary numbers: the **expectation** $\mathbb{E}[X]$ and the **variance** $\mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}X)^2]$. We then verify the most useful structural facts numerically: linearity (no independence required), the Jensen inequality for convex $\phi$, Markov, Chebyshev, and the weak law of large numbers.


In [ ]:
import numpy as np
np.random.seed(0)

# Discrete distribution on 5 outcomes.
values = np.array([-2.0, -1.0, 0.0, 1.0, 3.0])
probs  = np.array([0.10, 0.25, 0.30, 0.25, 0.10])
assert np.isclose(probs.sum(), 1.0)

EX_analytic   = float(np.sum(values * probs))
VarX_analytic = float(np.sum((values - EX_analytic)**2 * probs))

# Sample 10000 times.
samples = np.random.choice(values, size=10000, p=probs)
EX_emp   = float(samples.mean())
VarX_emp = float(samples.var(ddof=0))

print(f'E[X]  analytic = {EX_analytic:.6f}   empirical = {EX_emp:.6f}')
print(f'Var X analytic = {VarX_analytic:.6f}   empirical = {VarX_emp:.6f}')


## Linearity of expectation — without independence

$\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$ holds even when $X$ and $Y$ are *perfectly* dependent. Below, $Y = 2X + 1$, so $X$ determines $Y$ exactly; linearity is unaffected.


In [ ]:
import numpy as np
np.random.seed(0)

p = 0.3
n = 10000
X = (np.random.rand(n) < p).astype(np.float64)   # Bernoulli(p)
Y = 2.0 * X + 1.0                                # perfectly determined by X

EX     = X.mean()
EY     = Y.mean()
EX_pY  = (X + Y).mean()

print(f'E[X]        ~ {EX:.4f}    (analytic = {p})')
print(f'E[Y]        ~ {EY:.4f}    (analytic = {2*p + 1})')
print(f'E[X + Y]    ~ {EX_pY:.4f}  vs  E[X] + E[Y] = {EX + EY:.4f}')

# Covariance is maximal here, but linearity still holds.
cov_XY = np.cov(X, Y, ddof=0)[0, 1]
print(f'Cov(X, Y) = {cov_XY:.4f}   (X, Y are perfectly correlated)')


## Jensen's inequality

If $\phi$ is convex, then $\phi(\mathbb{E}[X]) \le \mathbb{E}[\phi(X)]$.

We test two convex functions: $\phi(x) = e^x$ on $X \sim \mathrm{Unif}\{-1,+1\}$, and $\phi(p) = -\log p$ on a categorical distribution (which connects directly to the entropy of Chapter 11).


In [ ]:
import numpy as np
np.random.seed(0)

# (1) phi(x) = exp(x), X uniform on {-1, +1}
X = np.random.choice([-1.0, 1.0], size=20000)
phi_of_EX = np.exp(X.mean())              # ~ exp(0) = 1
E_phi_X   = np.exp(X).mean()              # ~ (e + 1/e)/2 ~ 1.5431
print(f'phi(E[X]) = {phi_of_EX:.6f}')
print(f'E[phi(X)] = {E_phi_X:.6f}   (analytic = {(np.e + 1/np.e)/2:.6f})')
print(f'Jensen holds (E[phi(X)] >= phi(E[X])): {E_phi_X >= phi_of_EX}')

# (2) phi(p) = -log p, applied to outcomes drawn from a categorical p_true.
# E[-log p_true(X)] = entropy H(p_true).  By Jensen, H(p) >= -log E[p(X)] = -log sum p^2.
p_true = np.array([0.5, 0.25, 0.15, 0.10])
K = len(p_true)
draws  = np.random.choice(K, size=50000, p=p_true)
neg_log_p_X = -np.log(p_true[draws])
H_emp = neg_log_p_X.mean()
H_an  = -float(np.sum(p_true * np.log(p_true)))
jensen_lb = -np.log(np.sum(p_true**2))      # >= 0; tight only at uniform
print(f'\nEntropy H(p_true)  empirical = {H_emp:.6f}   analytic = {H_an:.6f}')
print(f'Jensen lower bound -log E[p(X)] = {jensen_lb:.6f}   (<= H)')


## Markov, Chebyshev, and the weak law of large numbers

Markov: $\mathbb{P}(X \ge a) \le \mathbb{E}[X]/a$ for $X \ge 0$.

Chebyshev: $\mathbb{P}(|X - \mu| \ge k\sigma) \le 1/k^2$.

Weak LLN: for i.i.d. $X_i$ with finite variance, $\bar X_n \to \mu$ in probability — proved directly via Chebyshev applied to $\bar X_n$, since $\mathrm{Var}(\bar X_n) = \sigma^2/n$.


In [ ]:
import numpy as np
np.random.seed(0)

# X_i ~ Uniform[0, 1], so mu = 1/2 and sigma^2 = 1/12.
mu, sigma2 = 0.5, 1.0/12.0
ns = np.unique(np.round(np.geomspace(10, 10000, num=12)).astype(int))

rows = []
for n in ns:
    Xn = np.random.uniform(0.0, 1.0, size=n)
    bar = float(Xn.mean())
    band = float(np.sqrt(sigma2 / n))           # one std-dev of bar X_n
    rows.append((n, bar, band))

print(f'{"n":>7} {"bar X_n":>12} {"|bar - mu|":>14} {"sigma/sqrt(n)":>16}')
for n, bar, band in rows:
    print(f'{n:>7d} {bar:>12.6f} {abs(bar-mu):>14.6f} {band:>16.6f}')

# Try to plot; fall back to the table above if matplotlib is unavailable.
try:
    import matplotlib.pyplot as plt
    ns_arr   = np.array([r[0] for r in rows], dtype=float)
    bars_arr = np.array([r[1] for r in rows], dtype=float)
    band_arr = np.array([r[2] for r in rows], dtype=float)
    plt.figure(figsize=(7, 4))
    plt.semilogx(ns_arr, bars_arr, 'o-', label=r'$\bar X_n$')
    plt.fill_between(ns_arr, mu - 2*band_arr, mu + 2*band_arr,
                     alpha=0.2, label=r'$\mu \pm 2\sigma/\sqrt{n}$ (Chebyshev / CLT band)')
    plt.axhline(mu, linestyle='--', label=r'$\mu = 1/2$')
    plt.xlabel('n'); plt.ylabel('sample mean'); plt.legend()
    plt.title('Weak law of large numbers, Uniform[0,1]')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f'matplotlib unavailable ({exc!r}); the table above is the fallback view.')


## Connection to LLMs

Training loss is $\mathcal{L}(\theta) = \mathbb{E}_{x \sim \mathcal{D}}[\ell(\theta; x)]$. Mini-batch SGD replaces this expectation by a sample mean of size $B$. By **linearity of expectation**, the SGD gradient is unbiased: $\mathbb{E}[\nabla \hat{\mathcal{L}}] = \nabla \mathcal{L}$. By the **variance-of-a-sum** identity (with independence within a batch), $\mathrm{Var}(\nabla \hat{\mathcal{L}}) = \Theta(1/B)$ — larger batches give a lower-variance estimator, and the **weak LLN** says we recover the true loss in probability as $B \to \infty$. **Jensen's inequality** powers the ELBO of variational inference (Chapter 22). Variance-reduction techniques for the SGD gradient are revisited in Chapter 13.


# Chapter 11 — Information theory: entropy, cross-entropy, KL

We use the natural log $\ln$ throughout. Self-information is $I(x) = -\ln p(x)$; entropy is $H(p) = \mathbb{E}_{x\sim p}[-\ln p(x)]$; cross-entropy is $H(p, q) = \mathbb{E}_{x \sim p}[-\ln q(x)]$; KL is $D_{\mathrm{KL}}(p \| q) = \mathbb{E}_{x \sim p}[\ln p(x)/q(x)]$. The two non-trivial facts proved in the chapter are: (1) **Gibbs**, $D_{\mathrm{KL}}(p \| q) \ge 0$; (2) the **decomposition** $H(p, q) = H(p) + D_{\mathrm{KL}}(p \| q)$. Everything else follows.


In [ ]:
import numpy as np

EPS = 1e-12

def entropy(p):
    p = np.asarray(p, dtype=float)
    # convention 0 ln 0 = 0 via clipping; contributions of zeros are zero anyway
    return float(-np.sum(np.where(p > 0, p * np.log(np.clip(p, EPS, 1.0)), 0.0)))

def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    return float(-np.sum(np.where(p > 0, p * np.log(np.clip(q, EPS, 1.0)), 0.0)))

def kl(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    ratio = np.log(np.clip(p, EPS, 1.0)) - np.log(np.clip(q, EPS, 1.0))
    return float(np.sum(np.where(p > 0, p * ratio, 0.0)))

# Two categoricals over 5 tokens.
p = np.array([0.05, 0.10, 0.50, 0.20, 0.15])
q = np.array([0.20, 0.20, 0.20, 0.20, 0.20])
assert np.isclose(p.sum(), 1.0) and np.isclose(q.sum(), 1.0)

Hp = entropy(p)
Hpq = cross_entropy(p, q)
Dpq = kl(p, q)
print(f'H(p)        = {Hp:.6f} nats')
print(f'H(p, q)     = {Hpq:.6f} nats')
print(f'D_KL(p||q)  = {Dpq:.6f} nats')
print(f'H(p) + D_KL = {Hp + Dpq:.6f} nats')
print(f'decomposition residual: {abs(Hpq - (Hp + Dpq)):.2e}')


## Gibbs' inequality

$D_{\mathrm{KL}}(p \| q) \ge 0$ for every pair of PMFs on the same support, with equality iff $p = q$. Proof by Jensen on the convex map $-\ln$ (Chapter 10). We sample 50 random Dirichlet pairs and confirm.


In [ ]:
rng = np.random.default_rng(0)
K = 5
alpha = np.ones(K)
ps = rng.dirichlet(alpha, size=50)
qs = rng.dirichlet(alpha, size=50)

kls = np.array([kl(p_i, q_i) for p_i, q_i in zip(ps, qs)])
self_kls = np.array([kl(p_i, p_i) for p_i in ps])

print(f'min  D_KL(p_i || q_i) over 50 pairs: {kls.min():.6f}')
print(f'mean D_KL(p_i || q_i) over 50 pairs: {kls.mean():.6f}')
print(f'all D_KL >= 0?                       {bool(np.all(kls >= -1e-12))}')
print(f'max |D_KL(p_i || p_i)|:              {np.max(np.abs(self_kls)):.2e}')


## Maximum entropy

On a finite support of size $K$, $H(p) \le \ln K$ with equality iff $p$ is uniform. Proof: $D_{\mathrm{KL}}(p \| u) = \ln K - H(p) \ge 0$ by Gibbs.


In [ ]:
K = 8
u = np.full(K, 1.0 / K)
Hu = entropy(u)
print(f'H(uniform on {K}) = {Hu:.6f} nats')
print(f'ln {K}            = {np.log(K):.6f} nats')
print(f'difference        = {abs(Hu - np.log(K)):.2e}')

rng = np.random.default_rng(0)
ps = rng.dirichlet(np.ones(K), size=100)
Hs = np.array([entropy(p_i) for p_i in ps])
print(f'\n100 random non-uniform p\'s on K={K}:')
print(f'  max H(p) = {Hs.max():.6f}  (must be < ln K = {np.log(K):.6f})')
print(f'  all H(p) < ln K? {bool(np.all(Hs < np.log(K) - 1e-12))}')


## Mutual information

$I(X; Y) = H(X) + H(Y) - H(X, Y) = D_{\mathrm{KL}}(p_{X,Y} \| p_X \otimes p_Y)$. Both formulas must agree, and both must vanish when $X, Y$ are independent.


In [ ]:
# Correlated joint on {0,1}^2: P(X=Y) is large.
# Rows = X in {0,1}, cols = Y in {0,1}.
P = np.array([[0.40, 0.10],
              [0.10, 0.40]])
assert np.isclose(P.sum(), 1.0)

pX = P.sum(axis=1)
pY = P.sum(axis=0)
P_indep = np.outer(pX, pY)

H_X  = entropy(pX)
H_Y  = entropy(pY)
H_XY = entropy(P.flatten())

I_via_entropies = H_X + H_Y - H_XY
I_via_kl        = kl(P.flatten(), P_indep.flatten())

print('Correlated joint:')
print(f'  H(X)        = {H_X:.6f}')
print(f'  H(Y)        = {H_Y:.6f}')
print(f'  H(X, Y)     = {H_XY:.6f}')
print(f'  I via H     = {I_via_entropies:.6f}')
print(f'  I via KL    = {I_via_kl:.6f}')
print(f'  agreement   = {abs(I_via_entropies - I_via_kl):.2e}')

# Independent joint: I should be 0.
P_prod = P_indep
I_prod = (entropy(P_prod.sum(axis=1)) + entropy(P_prod.sum(axis=0))
          - entropy(P_prod.flatten()))
I_prod_kl = kl(P_prod.flatten(), np.outer(P_prod.sum(axis=1), P_prod.sum(axis=0)).flatten())
print('\nIndependent joint p_X \u2297 p_Y:')
print(f'  I via H     = {I_prod:.2e}')
print(f'  I via KL    = {I_prod_kl:.2e}')


**Takeaway.** Cross-entropy $H(p_{\mathrm{data}}, p_\theta)$ is the next-token loss; minimizing it minimizes $D_{\mathrm{KL}}(p_{\mathrm{data}} \| p_\theta)$ since $H(p_{\mathrm{data}})$ is $\theta$-independent (Chapter 17, 25). KL appears again as the trust-region penalty in RLHF / DPO (Chapter 28). Entropy of the model's softmax controls sampling diversity at decoding time.


# Chapter 12 — Statistical inference: likelihood, MLE, ERM, bias-variance

We treat data as iid samples from a parametric family $\{p_\theta\}$ and use the **likelihood** to pick the best $\theta$. We then verify three claims numerically: (i) MLE for a Gaussian mean equals the sample mean; (ii) MLE for a categorical model equals the empirical distribution and minimizes cross-entropy; (iii) MSE = Bias$^2$ + Variance.


In [ ]:
import numpy as np

# --- Gaussian MLE: simulate N(mu=2, sigma=1), compute log-likelihood on a mu-grid ---
np.random.seed(0)
mu_true, sigma = 2.0, 1.0
n = 1000
X = np.random.normal(mu_true, sigma, size=n)

def log_lik_gauss(mu, X, sigma=1.0):
    return -0.5 * n * np.log(2*np.pi*sigma**2) - 0.5/sigma**2 * np.sum((X - mu)**2)

grid = np.linspace(1.0, 3.0, 4001)
ll = np.array([log_lik_gauss(m, X, sigma) for m in grid])
mu_hat_grid = grid[int(np.argmax(ll))]
mu_hat_closed = X.mean()

print(f'sample mean (closed form MLE): {mu_hat_closed:.6f}')
print(f'argmax over grid             : {mu_hat_grid:.6f}')
print(f'|difference|                 : {abs(mu_hat_grid - mu_hat_closed):.4e}')
assert abs(mu_hat_grid - mu_hat_closed) < 1e-3


## MLE = minimum cross-entropy (categorical case)

For a categorical model with classes $\{1,\dots,K\}$, the log-likelihood divided by $n$ is $\sum_k \hat p_n(k)\log p_\theta(k) = -H(\hat p_n, p_\theta)$. Maximizing the likelihood is therefore identical to minimizing cross-entropy, and the unique minimizer (subject to $\sum_k \theta_k = 1$) is $\theta = \hat p_n$ — provable by Lagrange multipliers, verified here numerically.


In [ ]:
import numpy as np

np.random.seed(0)
K = 4
p_true = np.array([0.1, 0.2, 0.3, 0.4])
n = 500
samples = np.random.choice(K, size=n, p=p_true)
p_hat = np.bincount(samples, minlength=K) / n
print(f'empirical distribution p_hat: {p_hat}')

def cross_entropy(p, q, eps=1e-12):
    return -np.sum(p * np.log(q + eps))

# Sweep theta on a small grid around p_hat (a tiny convex perturbation toward each vertex)
best_H, best_theta = np.inf, None
for alpha in np.linspace(0.0, 0.5, 11):
    for k in range(K):
        e_k = np.zeros(K); e_k[k] = 1.0
        theta = (1 - alpha) * p_hat + alpha * e_k
        H = cross_entropy(p_hat, theta)
        if H < best_H:
            best_H, best_theta = H, theta

H_at_phat = cross_entropy(p_hat, p_hat)
print(f'H(p_hat, p_hat)         = {H_at_phat:.6f}')
print(f'min H over grid         = {best_H:.6f}')
print(f'argmin theta over grid  = {best_theta}')
assert np.allclose(best_theta, p_hat)
assert abs(best_H - H_at_phat) < 1e-9


## Bias-variance decomposition

We estimate $\mu = 0$ for $X_i \sim \mathcal{N}(0,1)$ with two estimators:
- $\hat\mu_1 = \bar X_n$ (unbiased, variance $1/n$);
- $\hat\mu_2 = (\bar X_n + 1)/2$ (biased toward $1/2$, variance $1/(4n)$).

Run $T = 1000$ trials at $n = 10$, compute empirical $\mathrm{Bias}^2$, $\mathrm{Var}$, $\mathrm{MSE}$, and verify $\mathrm{MSE} = \mathrm{Bias}^2 + \mathrm{Var}$.


In [ ]:
import numpy as np

np.random.seed(0)
T, n, mu_true = 1000, 10, 0.0
samples = np.random.normal(mu_true, 1.0, size=(T, n))
xbar = samples.mean(axis=1)
mu1 = xbar
mu2 = (xbar + 1.0) / 2.0

def report(name, est, mu_true):
    bias = est.mean() - mu_true
    var = est.var(ddof=0)
    mse = ((est - mu_true)**2).mean()
    print(f'{name:>10s}: bias^2={bias**2:.5f}  var={var:.5f}  '
          f'bias^2+var={bias**2+var:.5f}  MSE={mse:.5f}')
    assert abs((bias**2 + var) - mse) < 1e-12
    return bias**2, var, mse

report('mu1 = Xbar', mu1, mu_true)
report('mu2 biased', mu2, mu_true)
print('Decomposition Bias^2 + Var = MSE verified for both estimators.')


## ERM as a generalization of MLE

Replace $-\log p_\theta(x)$ by an arbitrary loss $\ell(\theta;x)$ and minimize the empirical mean. Linear regression with squared loss $\ell(w; (x,y)) = (y - wx)^2$ is the canonical example; the ERM solution is the **normal equation** $\hat w = (\sum_i x_i y_i)/(\sum_i x_i^2)$.


In [ ]:
import numpy as np

np.random.seed(0)
w_true, sigma_eps, n = 3.0, 0.5, 100
x = np.random.normal(0.0, 1.0, size=n)
eps = np.random.normal(0.0, sigma_eps, size=n)
y = w_true * x + eps

# Normal equation in 1D: w_hat = (x . y) / (x . x)
w_hat = float(np.dot(x, y) / np.dot(x, x))
emp_risk = float(np.mean((y - w_hat * x)**2))
print(f'true w        : {w_true}')
print(f'ERM estimate  : {w_hat:.6f}')
print(f'|w_hat - w*|  : {abs(w_hat - w_true):.4e}')
print(f'empirical risk: {emp_risk:.6f}  (noise variance {sigma_eps**2:.4f})')
assert abs(w_hat - w_true) < 0.1


## Connection to LLMs

An autoregressive language model factors $p_\theta(x_{1:T}) = \prod_t p_\theta(x_t \mid x_{<t})$. The pre-training loss is
$$-\frac{1}{N}\sum_{\text{seq}}\sum_t \log p_\theta(x_t \mid x_{<t}) \;=\; H(\hat p_n,\,p_\theta),$$
i.e. cross-entropy with the empirical token distribution. By Theorem 12.1 this is exactly MLE; by its corollary it is KL projection of the model onto the empirical corpus. Bias-variance (Theorem 12.3) then governs the under/overfitting trade-off explored in Chapter 27.


# Block C — Stochastic Optimization


## SGD: stochastic-approximation theorem; mini-batching; convergence sketch

Full-batch gradient descent (Chapter 7) requires a pass over every data point per step. For a corpus of $10^{13}$ tokens that is unaffordable. Stochastic gradient descent (SGD) replaces $\nabla F(\theta) = \mathbb{E}_\xi \nabla f(\theta;\xi)$ with a sample estimate $\hat g_B = \frac{1}{B}\sum_{j=1}^B \nabla f(\theta;\xi_j)$ and accepts noise in exchange for cheap steps.

We illustrate four facts from this chapter on tiny problems:
1. SGD is noisier than GD but still converges.
2. Variance of $\hat g_B$ scales as $1/B$.
3. On a smooth non-convex toy, $\frac{1}{T}\sum_t \|\nabla F(\theta_t)\|^2 = O(1/\sqrt{T})$.
4. Constant step plateaus at a noise floor; diminishing step (Robbins--Monro) converges.


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)
n = 1000
y = np.random.normal(loc=3.0, scale=1.0, size=n)  # F(theta) = (1/n) sum (theta - y_i)^2; min at mean(y)
theta_star = y.mean()

def loss(theta):
    return float(np.mean((theta - y) ** 2))

def full_grad(theta):
    return 2.0 * (theta - y.mean())

def stoch_grad(theta, B=1, rng=None):
    rng = rng or np.random
    idx = rng.randint(0, n, size=B)
    return 2.0 * (theta - y[idx].mean())

T = 200
eta = 0.05
theta_gd = 0.0
loss_gd = []
for t in range(T):
    loss_gd.append(loss(theta_gd))
    theta_gd -= eta * full_grad(theta_gd)

rng = np.random.RandomState(0)
theta_sgd = 0.0
loss_sgd = []
for t in range(T):
    loss_sgd.append(loss(theta_sgd))
    theta_sgd -= eta * stoch_grad(theta_sgd, B=1, rng=rng)

print(f'optimum theta* = {theta_star:.4f}')
print(f'GD final theta  = {theta_gd:.4f}, loss = {loss_gd[-1]:.4f}')
print(f'SGD final theta = {theta_sgd:.4f}, loss = {loss_sgd[-1]:.4f}')
if HAVE_MPL:
    plt.figure()
    plt.plot(loss_gd, label='Full-batch GD')
    plt.plot(loss_sgd, label='SGD (B=1)', alpha=0.7)
    plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.title('GD vs SGD on least squares')
    plt.show()


### Variance reduction by batching

Theorem 13.1: $\mathbb{E}\|\hat g_B - \nabla F\|^2 \leq \sigma^2/B$. We verify this on the same least-squares problem at $\theta = 0$ by drawing $200$ random batches for each $B$ and measuring the empirical variance of the resulting gradient estimates.


In [ ]:
import numpy as np
np.random.seed(0)
n = 1000
y = np.random.normal(loc=3.0, scale=1.0, size=n)
true_grad_at_0 = 2.0 * (0.0 - y.mean())

rng = np.random.RandomState(0)
Bs = [1, 8, 64, 256]
K = 200  # number of random batches per B
print(f'{"B":>5}  {"empirical Var":>14}  {"sigma^2 / B":>14}  {"ratio":>8}')
sigma2 = 4.0 * np.var(y)  # population variance of g(theta=0;xi) = 2*(0 - y_i)
for B in Bs:
    estimates = np.array([2.0 * (0.0 - y[rng.randint(0, n, size=B)].mean()) for _ in range(K)])
    var_emp = float(np.var(estimates))
    pred = sigma2 / B
    print(f'{B:>5}  {var_emp:>14.5f}  {pred:>14.5f}  {var_emp/pred:>8.3f}')


### SGD on a smooth non-convex toy

Let $F(\theta) = \theta^2/2 + 0.5\sin(5\theta)$. This is $L$-smooth (with $L = 1 + 12.5 = 13.5$, since $|F''| \leq 1 + 12.5$) but non-convex. Theorem 13.2 predicts the running average of $\|\nabla F(\theta_t)\|^2$ decays like $1/\sqrt{T}$.


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)

def grad_F(theta):
    return theta + 2.5 * np.cos(5.0 * theta)

T = 5000
eta = 0.02
noise_sigma = 1.0
rng = np.random.RandomState(0)
theta = 2.0
g2 = []
for t in range(T):
    g = grad_F(theta) + noise_sigma * rng.randn()
    g2.append(grad_F(theta) ** 2)
    theta -= eta * g

g2 = np.array(g2)
running_avg = np.cumsum(g2) / np.arange(1, T + 1)
ts = np.arange(1, T + 1)
ref = running_avg[10] * np.sqrt(11) / np.sqrt(ts)
print(f'avg ||grad F||^2 at T=100 : {running_avg[99]:.4f}')
print(f'avg ||grad F||^2 at T=1000: {running_avg[999]:.4f}')
print(f'avg ||grad F||^2 at T=5000: {running_avg[-1]:.4f}')
print(f'ratio T=100/T=5000        : {running_avg[99]/running_avg[-1]:.2f}  (expected ~ sqrt(50) = 7.07)')
if HAVE_MPL:
    plt.figure()
    plt.loglog(ts, running_avg, label='running avg ||grad F||^2')
    plt.loglog(ts, ref, '--', label='reference ~ 1/sqrt(T)')
    plt.xlabel('T'); plt.ylabel('avg sq grad'); plt.legend(); plt.title('SGD: 1/sqrt(T) decay')
    plt.show()


### Constant vs diminishing step (Robbins--Monro)

Strongly-convex quadratic $F(\theta) = \frac{1}{2}\theta^2$, with additive noise on the gradient. A constant step $\eta$ leaves the iterates orbiting the optimum at a noise floor $\sim \eta \sigma^2$; the diminishing schedule $\eta_t = c/(t+1)$ satisfies $\sum \eta_t = \infty, \sum \eta_t^2 < \infty$ and converges (slowly).


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)
T = 5000
sigma = 1.0
rng = np.random.RandomState(0)

# constant step
eta_c = 0.1
theta = 2.0
dist_const = []
for t in range(T):
    g = theta + sigma * rng.randn()
    theta -= eta_c * g
    dist_const.append(theta ** 2)

# diminishing step
rng = np.random.RandomState(0)
c = 1.0
theta = 2.0
dist_dim = []
for t in range(T):
    eta_t = c / (t + 1)
    g = theta + sigma * rng.randn()
    theta -= eta_t * g
    dist_dim.append(theta ** 2)

dist_const = np.array(dist_const)
dist_dim = np.array(dist_dim)
tail_const = float(np.mean(dist_const[-500:]))
tail_dim = float(np.mean(dist_dim[-500:]))
print(f'constant step eta=0.1  tail E[(theta-theta*)^2] = {tail_const:.5f}  (noise floor ~ eta*sigma^2/2 = {eta_c*sigma**2/2:.5f})')
print(f'diminishing step c/(t+1) tail E[(theta-theta*)^2] = {tail_dim:.5f}  (predicted O(1/T))')
if HAVE_MPL:
    plt.figure()
    ts = np.arange(1, T + 1)
    plt.loglog(ts, np.maximum(dist_const, 1e-8), label='constant eta=0.1')
    plt.loglog(ts, np.maximum(dist_dim, 1e-8), label='eta_t = 1/(t+1)')
    plt.loglog(ts, 1.0/ts, '--', label='1/T reference')
    plt.xlabel('t'); plt.ylabel('(theta_t - theta*)^2'); plt.legend(); plt.title('Constant vs diminishing step')
    plt.show()


**Takeaway.** The four experiments above match each chapter theorem: SGD converges (noisier than GD); $\mathrm{Var}(\hat g_B) \propto 1/B$; the running average of $\|\nabla F\|^2$ decays like $1/\sqrt{T}$ on a smooth non-convex problem; and only Robbins--Monro step sizes drive the iterates to the true optimum in the presence of persistent gradient noise. Pre-training an LLM (Chapter 23) inherits this entire picture; AdamW (Chapter 14) layers momentum and adaptive scaling on top.


# Chapter 14 --- Momentum, RMSProp, AdamW: derivation + bias-correction proof

Plain SGD (Chapter 13) suffers on ill-conditioned losses: it oscillates across stiff directions and crawls along flat ones. **Momentum** smooths the trajectory by giving the iterate inertia. We start with the canonical demo: a quadratic with condition number $\kappa = 100$.

$F(x, y) = \tfrac{1}{2}(x^2 + 100\, y^2)$, optimum at the origin.

In [ ]:
import numpy as np

def grad_quad(theta):
    return np.array([theta[0], 100.0 * theta[1]])

def run_sgd(eta=0.018, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); traj = [theta.copy()]
    for _ in range(T):
        theta = theta - eta * grad_quad(theta)
        traj.append(theta.copy())
    return np.array(traj)

def run_momentum(eta=0.018, beta=0.9, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); v = np.zeros_like(theta); traj = [theta.copy()]
    for _ in range(T):
        v = beta * v + grad_quad(theta)
        theta = theta - eta * v
        traj.append(theta.copy())
    return np.array(traj)

tr_sgd = run_sgd(); tr_mom = run_momentum()
print('final |theta| -- SGD     :', np.linalg.norm(tr_sgd[-1]))
print('final |theta| -- momentum:', np.linalg.norm(tr_mom[-1]))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.plot(tr_sgd[:, 0], tr_sgd[:, 1], 'o-', ms=3, label='SGD')
    plt.plot(tr_mom[:, 0], tr_mom[:, 1], 's-', ms=3, label='Momentum (beta=0.9)')
    plt.scatter([0], [0], c='k', marker='*', s=80, label='optimum')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('Ill-conditioned quadratic')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## RMSProp --- per-coordinate adaptive rescaling

RMSProp keeps an EMA of squared gradients and divides each step by its square root, so coordinates with historically large gradients receive smaller effective learning rates. On our quadratic the $y$-coordinate has gradient magnitude $100\times$ larger than $x$; RMSProp evens this out automatically.

In [ ]:
import numpy as np

def grad_quad(theta):
    return np.array([theta[0], 100.0 * theta[1]])

def run_rmsprop(eta=0.05, beta2=0.9, eps=1e-8, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); v = np.zeros_like(theta); traj = [theta.copy()]
    for _ in range(T):
        g = grad_quad(theta)
        v = beta2 * v + (1 - beta2) * g * g
        theta = theta - eta * g / (np.sqrt(v) + eps)
        traj.append(theta.copy())
    return np.array(traj)

tr_rms = run_rmsprop()
print('final theta (RMSProp):', tr_rms[-1])
print('final |theta|        :', np.linalg.norm(tr_rms[-1]))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.plot(tr_rms[:, 0], tr_rms[:, 1], 'd-', ms=3, color='C2', label='RMSProp')
    plt.scatter([0], [0], c='k', marker='*', s=80)
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('RMSProp trajectory')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## Adam from scratch --- with bias correction

Adam = momentum (first moment) + RMSProp (second moment) + bias correction. We implement it in $\sim$ 25 lines and verify convergence on $F(\theta) = \tfrac{1}{2}\|\theta - \theta^\star\|^2$ with $\theta^\star = (3, -2)$. The gradient is $g = \theta - \theta^\star$, constant in expectation along a single trajectory; we will see $\hat m_t$ track this gradient closely from $t=1$.

In [ ]:
import numpy as np
np.random.seed(0)

theta_star = np.array([3.0, -2.0])
def grad(theta):
    return theta - theta_star

def adam(theta0, T=400, eta=0.1, b1=0.9, b2=0.999, eps=1e-8, verbose_steps=5):
    theta = np.array(theta0, dtype=float)
    m = np.zeros_like(theta); v = np.zeros_like(theta)
    for t in range(1, T + 1):
        g = grad(theta)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        theta = theta - eta * m_hat / (np.sqrt(v_hat) + eps)
        if t <= verbose_steps:
            print(f't={t}: g={g}, m_hat={m_hat}, v_hat={v_hat}')
    return theta

theta_final = adam(theta0=(0.0, 0.0))
print('theta_final:', theta_final)
print('theta_star :', theta_star)
print('error norm :', np.linalg.norm(theta_final - theta_star))

## Bias-correction proof --- empirical verification

Theorem: with $\mathbb{E}[g_t] = g$ and $m_0 = 0$, the EMA satisfies $\mathbb{E}[m_t] = (1 - \beta_1^t)\, g$. Without correction, $m_1 \approx 0.1\, g$ at $\beta_1 = 0.9$. The corrected estimate $\hat m_t = m_t/(1 - \beta_1^t)$ is unbiased for every $t \geq 1$. We simulate.

In [ ]:
import numpy as np
np.random.seed(0)

T = 100; b1 = 0.9; g_true = 1.0; sigma = 0.5
n_trials = 5000
g_samples = np.random.normal(g_true, sigma, size=(n_trials, T))

M = np.zeros((n_trials, T))
M[:, 0] = (1 - b1) * g_samples[:, 0]
for t in range(1, T):
    M[:, t] = b1 * M[:, t - 1] + (1 - b1) * g_samples[:, t]

denom = 1 - b1 ** np.arange(1, T + 1)
M_hat = M / denom

mean_m   = M.mean(axis=0)
mean_mh  = M_hat.mean(axis=0)
print('first 6 E[m_t]    :', mean_m[:6].round(4))
print('first 6 E[m_hat_t]:', mean_mh[:6].round(4))
print('theory (1-b^t)*g  :', ((1 - b1 ** np.arange(1, 7)) * g_true).round(4))
try:
    import matplotlib.pyplot as plt
    ts = np.arange(1, T + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(ts, mean_m, label='E[m_t] (uncorrected)')
    plt.plot(ts, mean_mh, label='E[m_hat_t] (bias-corrected)')
    plt.axhline(g_true, color='k', linestyle='--', label='target g = 1')
    plt.xlabel('t'); plt.ylabel('estimate'); plt.legend()
    plt.title('Bias correction removes the early-step underestimate')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## AdamW vs Adam + L2 --- a tiny logistic regression

For SGD, $L_2$ regularization and decoupled weight decay coincide. For Adam they do not: the $L_2$ gradient $\lambda\theta$ is rescaled by $1/\sqrt{\hat v_t}$, so coordinates with large historical gradients get decayed less than intended. AdamW restores uniform shrinkage by applying $\lambda\theta$ outside the adaptive rescaling. We compare the two on a 2D synthetic binary classification.

In [ ]:
import numpy as np
np.random.seed(0)

n = 200
X = np.random.randn(n, 2)
true_w = np.array([2.0, -1.5])
logits = X @ true_w
y = (logits + 0.3 * np.random.randn(n) > 0).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def loss_and_grad(w, X, y):
    z = X @ w
    p = sigmoid(z)
    loss = -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    g = X.T @ (p - y) / len(y)
    return loss, g

def train(mode, T=2000, eta=0.05, b1=0.9, b2=0.999, eps=1e-8, lam=0.1):
    w = np.zeros(2); m = np.zeros(2); v = np.zeros(2)
    for t in range(1, T + 1):
        loss, g = loss_and_grad(w, X, y)
        if mode == 'adam_l2':
            g = g + lam * w  # L2 enters the gradient
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1**t); v_hat = v / (1 - b2**t)
        if mode == 'adamw':
            w = w - eta * (m_hat / (np.sqrt(v_hat) + eps) + lam * w)
        else:
            w = w - eta * m_hat / (np.sqrt(v_hat) + eps)
    final_loss, _ = loss_and_grad(w, X, y)
    return w, final_loss

w_l2, loss_l2     = train('adam_l2')
w_aw, loss_aw     = train('adamw')
print(f'Adam + L2 : loss={loss_l2:.4f}  ||w||={np.linalg.norm(w_l2):.4f}  w={w_l2}')
print(f'AdamW     : loss={loss_aw:.4f}  ||w||={np.linalg.norm(w_aw):.4f}  w={w_aw}')
print('Decoupled decay yields the smaller-norm solution (stronger effective regularization).')

# Block D — Neural Networks


<!-- CHAPTER 15 START -->
## Chapter 15: MLPs as compositional functions; universal approximation

_Stub — will be filled by Phase 2._
<!-- CHAPTER 15 END -->

<!-- CHAPTER 16 START -->
## Chapter 16: Activation functions: ReLU/GELU/softmax with derivatives

_Stub — will be filled by Phase 2._
<!-- CHAPTER 16 END -->

<!-- CHAPTER 17 START -->
## Chapter 17: Loss functions: MSE, cross-entropy; gradients from first principles

_Stub — will be filled by Phase 2._
<!-- CHAPTER 17 END -->

<!-- CHAPTER 18 START -->
## Chapter 18: Backpropagation: chain rule applied; reverse-mode AD as a graph algorithm

_Stub — will be filled by Phase 2._
<!-- CHAPTER 18 END -->

# Block E — Sequence Models and Attention


<!-- CHAPTER 19 START -->
## Chapter 19: Embeddings: token to vector; lookup as a linear map; weight tying

_Stub — will be filled by Phase 2._
<!-- CHAPTER 19 END -->

<!-- CHAPTER 20 START -->
## Chapter 20: RNN intuition; vanishing-gradient proof; why we need attention

_Stub — will be filled by Phase 2._
<!-- CHAPTER 20 END -->

<!-- CHAPTER 21 START -->
## Chapter 21: Scaled dot-product attention: derivation, softmax-temperature analysis

_Stub — will be filled by Phase 2._
<!-- CHAPTER 21 END -->

<!-- CHAPTER 22 START -->
## Chapter 22: Multi-head attention: parallel heads as concat-then-project; complexity

_Stub — will be filled by Phase 2._
<!-- CHAPTER 22 END -->

<!-- CHAPTER 23 START -->
## Chapter 23: Transformer block: residual + LayerNorm/RMSNorm + FFN + attention; gradient-flow argument

_Stub — will be filled by Phase 2._
<!-- CHAPTER 23 END -->

<!-- CHAPTER 24 START -->
## Chapter 24: Positional encoding: sinusoidal derivation, RoPE construction

_Stub — will be filled by Phase 2._
<!-- CHAPTER 24 END -->

# Block F — Pre-training


<!-- CHAPTER 25 START -->
## Chapter 25: Causal masking; next-token prediction loss as MLE on the empirical distribution

_Stub — will be filled by Phase 2._
<!-- CHAPTER 25 END -->

<!-- CHAPTER 26 START -->
## Chapter 26: Tokenization: BPE algorithm; greedy merge correctness

_Stub — will be filled by Phase 2._
<!-- CHAPTER 26 END -->

<!-- CHAPTER 27 START -->
## Chapter 27: Pre-training pipeline: AdamW + warmup + cosine decay + gradient clipping; tiny-GPT training run

_Stub — will be filled by Phase 2._
<!-- CHAPTER 27 END -->

# Block G — Post-training


<!-- CHAPTER 28 START -->
## Chapter 28: SFT, RLHF (PPO/GRPO), and DPO; train + post-train a tiny GPT

_Stub — will be filled by Phase 2._
<!-- CHAPTER 28 END -->